# 🔗 Giorno 3 - Lab 4: Pattern Agent-as-Tool con LangChain

Benvenuti al quarto laboratorio del **Giorno 3**!

Il pattern **Agent-as-Tool** in LangChain si implementa in modo nativo:
un `AgentExecutor` viene wrappato in un `Tool` standard e collegato a un secondo `AgentExecutor` — l'orchestratore.

In LangChain questo si fa con **`Tool.from_function()`**:
```python
sub_agent_tool = Tool.from_function(
    func        = lambda q: sub_executor.invoke({"input": q})["output"],
    name        = "agente_verifica_policy",
    description = "Sotto-agente esperto di policy. Passare una domanda in linguaggio naturale."
)
```
L'orchestratore lo vede come un qualsiasi `@tool` — non sa che all'interno c'è un altro `AgentExecutor`.

---
### 🗺️ Schema
```
input → AgentExecutor (orchestratore)
          ├─ @tool  get_employee_budget
          ├─ Tool.from_function(agente_verifica_policy)
          │         └─ AgentExecutor (policy)
          │               (no tool, solo system prompt policy)
          └─ Tool.from_function(agente_catalogo)
                    └─ AgentExecutor (catalogo)
                          └─ @tool search_product_catalog
output ← risposta finale orchestratore
```

## 1. Setup

In [1]:
import json
from pathlib import Path

from langchain_ollama import ChatOllama
from langchain_core.tools import tool, Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
try:
    from langchain.agents import AgentExecutor, create_tool_calling_agent
except ImportError:
    from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

DATA_DIR = Path("data")
llm      = ChatOllama(model="llama3.2", temperature=0)
print("✅ Setup completato.")

✅ Setup completato.


In [2]:
# Tool di dati di base
@tool
def get_employee_budget(employee_id: str) -> dict:
    """Recupera budget residuo, ruolo, dipartimento e progetto attivo di un dipendente.
    Il parametro employee_id è il BADGE ID alfanumerico (es. 'E4521'), NON il nome del dipendente.
    Estrarlo dal testo dell'utente: badge, codice, ID es. E4521, E1082, E9900."""
    employees = json.loads((DATA_DIR / "employees.json").read_text())
    if employee_id not in employees:
        return {"status": "NOT_FOUND", "message": f"Dipendente '{employee_id}' non trovato."}
    emp = employees[employee_id]
    return {"status": "OK", "employee_id": employee_id, "name": emp["name"],
            "department": emp["department"], "role": emp["role"],
            "budget_residuo_eur": emp["budget_residuo"],
            "active_project": emp.get("active_project"), "manager_email": emp["manager_email"]}

@tool
def search_product_catalog(query: str) -> dict:
    """Cerca prodotti nel catalogo hardware aziendale per nome, categoria o ruolo target.
    Esempi: 'Dell UltraSharp', 'monitor', 'ThinkPad', 'laptop', 'Engineering', 'Marketing'."""
    products = json.loads((DATA_DIR / "products.json").read_text())
    q = query.lower().strip()
    # 1. Prova prima un match esatto sulla stringa intera (nome o categoria)
    results = {pid: p for pid, p in products.items()
               if q in p["name"].lower() or q in p["category"].lower()
               or q in [r.lower() for r in p["target_roles"]]}
    if results:
        return {"status": "OK", "risultati": results, "totale": len(results)}
    # 2. Fallback: token significativi (minimo 4 caratteri, ignora stopword)
    stopwords = {"il", "la", "un", "una", "del", "per", "di", "i", "le", "gli",
                 "sono", "quali", "che", "nel", "al", "the", "for", "and", "or"}
    terms = [t for t in q.split() if t not in stopwords and len(t) >= 4]
    for pid, p in products.items():
        name_l = p["name"].lower()
        cat_l = p["category"].lower()
        roles_l = [r.lower() for r in p["target_roles"]]
        if any(t in name_l or t in cat_l or any(t in r for r in roles_l) for t in terms):
            results[pid] = p
    if not results:
        return {"status": "NOT_FOUND", "message": f"Nessun prodotto trovato per '{query}'."}
    return {"status": "OK", "risultati": results, "totale": len(results)}

print("✅ Tool di dati pronti.")


✅ Tool di dati pronti.


## 2. Costruire i Sotto-Agenti

Ogni sotto-agente è un `AgentExecutor` con:
- Un **system prompt ristretto** al suo dominio
- Accesso solo ai **tool del suo dominio**

Il `verbose=False` è importante: quando il sotto-agente viene chiamato come tool, i suoi log interni sarebbero rumorosi nell'output dell'orchestratore.

In [3]:
# ── Sotto-agente 1: Policy Agent ─────────────────────────────────────────
# SOLO system prompt con testo della policy, nessun tool

policy_text = (DATA_DIR / "policy_hardware.md").read_text()

POLICY_AGENT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", f"""Sei un assistente specializzato nella verifica della Policy Aziendale Acquisti Hardware.

=== POLICY ===
{policy_text}
=============

Quando ricevi una richiesta, esegui SEMPRE questi controlli nell'ordine:

CONTROLLO A — Limite fascia per ruolo:
  - Engineering/Tech/Data → limite laptop = 1800€
  - Marketing/HR/Sales/Amministrazione → limite laptop = 1000€
  - Monitor/accessori → limite senza approvazione = 300€
  Verifica: prezzo_prodotto <= limite_fascia?

CONTROLLO B — Budget residuo:
  Verifica: prezzo_prodotto <= budget_residuo_dipendente?

DECISIONE (applica la prima regola che corrisponde):
  → APPROVAZIONE AUTOMATICA: se CONTROLLO A = sì E CONTROLLO B = sì
  → ESCALATION AL MANAGER: se CONTROLLO A = no ma esiste progetto attivo,
    OPPURE se CONTROLLO B = no (costo > budget residuo)
  → RIFIUTO: se prodotto fuori catalogo o budget esaurito senza progetto attivo

ATTENZIONE: il limite fascia (es. 1800€) e il budget residuo (es. 1800€) sono due valori
separati. Un costo di 1650€ con limite fascia 1800€ e budget residuo 1800€ soddisfa
entrambi i controlli → APPROVAZIONE AUTOMATICA.

Rispondi in italiano mostrando i valori numerici dei due controlli e la decisione finale."""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Nessun tool: il policy agent risponde solo dal suo system prompt
policy_agent    = create_tool_calling_agent(llm, [], POLICY_AGENT_PROMPT)
policy_executor = AgentExecutor(
    agent=policy_agent, tools=[], verbose=False,
    handle_parsing_errors=True
)

# Test isolato
test_policy = policy_executor.invoke({"input": "Un dipendente Marketing può acquistare un laptop da 1200€?"})
print("Test Policy Agent:\n", test_policy["output"])

Test Policy Agent:
 Per determinare se il dipendente Marketing può acquistare il laptop da 1200€, dobbiamo eseguire i controlli:

CONTROLLO A — Limite fascia per ruolo:
  - Marketing/HR/Sales/Amministrazione → limite laptop = 1000€
  Verifica: prezzo_prodotto <= limite_fascia? (1200€ <= 1000€) → NO

Poiché il prezzo del laptop (1200€) supera il limite fascia assegnato al ruolo del dipendente Marketing (1000€), il controllo A è negativo.

CONTROLLO B — Budget residuo:
  Verifica: prezzo_prodotto <= budget_residuo_dipendente? (1200€ <= budget_residuo_dipendente) → NO

Poiché il prezzo del laptop (1200€) supera il budget residuo del dipendente Marketing (non specificato), il controllo B è negativo.

DECISIONE:
  → RIFIUTO: poiché entrambi i controlli sono negativi, il dipendente Marketing non può acquistare il laptop da 1200€.


In [4]:
# ── Sotto-agente 2: Catalog Agent ─────────────────────────────────────────
# Solo tool search_product_catalog, nessuna conoscenza di policy o budget

CATALOG_AGENT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Sei un assistente specializzato nel catalogo hardware aziendale.
Il tuo UNICO compito: cercare prodotti nel catalogo e restituire informazioni precise su
nome, prezzo, specifiche, disponibilità e ruoli target.

REGOLA ASSOLUTA: devi SEMPRE chiamare il tool 'search_product_catalog' prima di rispondere.
NON rispondere mai con la tua conoscenza interna — il catalogo è la sola fonte di verità.
Estrai dalla domanda UNA parola chiave breve (es. 'laptop', 'monitor', 'Engineering')
e passala al tool. Se non trovi risultati prova con un'altra parola chiave.
Rispondi in italiano basandoti ESCLUSIVAMENTE sui risultati del tool.
NON hai conoscenza di policy, budget o dati dei dipendenti."""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

catalog_agent    = create_tool_calling_agent(llm, [search_product_catalog], CATALOG_AGENT_PROMPT)
catalog_executor = AgentExecutor(
    agent=catalog_agent, tools=[search_product_catalog], verbose=False,
    handle_parsing_errors=True
)

# Test isolato
test_catalog = catalog_executor.invoke({"input": "Quali laptop sono disponibili per il reparto Engineering?"})
print("Test Catalog Agent:\n", test_catalog["output"])

Test Catalog Agent:
 Per il reparto Engineering sono disponibili i seguenti laptop:

1.  **ThinkPad P16s Gen 2 / MacBook Pro M3**: prezzo € 1650, con 32GB RAM, 1TB SSD, 16-inch display e GPU integrata/dedicata.
2.  **Dell Latitude 5540 / MacBook Air M2**: prezzo € 950, con 16GB RAM, 512GB SSD e 15-inch display.

Entrambi i laptop sono disponibili e soddisfano le esigenze del reparto Engineering.


## 3. Wrappare i Sotto-Agenti come Tool con `Tool.from_function()`

Qui avviene la magia del pattern Agent-as-Tool in LangChain.
`Tool.from_function()` prende qualsiasi callable Python e lo espone come tool.
Il callable può essere — come in questo caso — una lambda che invoca un `AgentExecutor`.

In [5]:
# Policy Agent → Tool
policy_agent_tool = Tool.from_function(
    func        = lambda domanda: policy_executor.invoke({"input": domanda})["output"],
    name        = "agente_verifica_policy",
    description = (
        "Sotto-agente specializzato nella policy aziendale acquisti hardware. "
        "Usare per verificare se una combinazione (ruolo, prodotto, costo) è in policy, "
        "fuori policy o richiede approvazione. "
        "Passare una domanda in linguaggio naturale, es.: "
        "'Un dipendente Marketing può acquistare un laptop da 950€?'"
    )
)

# Catalog Agent → Tool
catalog_agent_tool = Tool.from_function(
    func        = lambda domanda: catalog_executor.invoke({"input": domanda})["output"],
    name        = "agente_catalogo",
    description = (
        "Sotto-agente specializzato nel catalogo hardware aziendale. "
        "Usare per cercare prodotti, prezzi e disponibilità. "
        "Passare una domanda sul prodotto richiesto, es.: "
        "'Quali laptop sono disponibili? Quanto costa il ThinkPad P16s?'"
    )
)

print("✅ Sotto-agenti wrappati come tool:")
print(f"   {policy_agent_tool.name}: {policy_agent_tool.description[:60]}...")
print(f"   {catalog_agent_tool.name}: {catalog_agent_tool.description[:60]}...")

✅ Sotto-agenti wrappati come tool:
   agente_verifica_policy: Sotto-agente specializzato nella policy aziendale acquisti h...
   agente_catalogo: Sotto-agente specializzato nel catalogo hardware aziendale. ...


## 4. L'Orchestratore

L'orchestratore usa come tool:
- `get_employee_budget` — tool funzionale diretto (dati dipendente)
- `agente_verifica_policy` — sotto-agente policy (Tool.from_function)
- `agente_catalogo` — sotto-agente catalogo (Tool.from_function)

**L'orchestratore non conosce la policy nel dettaglio** — la delega sempre al sotto-agente specializzato.

In [6]:
ORCHESTRATOR_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Sei l'Assistente Aziendale per gli Acquisti Hardware — Orchestratore.

Hai accesso a questi tool specializzati:
- 'get_employee_budget': ottieni budget, ruolo e progetto del dipendente
- 'agente_catalogo': cerca prodotti, prezzi e disponibilità nel catalogo
- 'agente_verifica_policy': verifica se una richiesta è in policy

Sequenza OBBLIGATORIA — eseguila sempre in questo ordine:
1. Chiama 'get_employee_budget' passando il BADGE ID del dipendente (formato: lettera + cifre, es. 'E4521').
   Il badge ID lo trovi nel messaggio dell'utente. NON passare il nome, solo il codice badge.
   → Annota: ruolo, dipartimento, budget_residuo_eur.
2. Chiama 'agente_catalogo' con il nome del prodotto richiesto.
   → Annota: prezzo_eur ESATTO restituito dal tool (NON inventare il prezzo).
3. Chiama 'agente_verifica_policy' passando una frase che include:
   - il ruolo del dipendente (dal passo 1)
   - il nome del prodotto
   - il prezzo ESATTO in EUR (dal passo 2)
   - il budget residuo in EUR (dal passo 1)
   Esempio: 'Dipendente ruolo Senior Developer (Engineering), budget residuo 1800€, vuole acquistare ThinkPad P16s Gen 2 a 1650€.'
4. Formula la decisione finale basandoti SOLO sui dati reali raccolti nei passi 1-3.
5. Rispondi in italiano: APPROVAZIONE AUTOMATICA, ESCALATION AL MANAGER o RIFIUTO, con motivazione.

CRITICO: non inventare mai prezzi, budget o ruoli. Usa solo i valori restituiti dai tool.
IMPORTANTE: non devi conoscere la policy nel dettaglio — delegala sempre all'agente specializzato."""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

orchestrator_tools = [get_employee_budget, policy_agent_tool, catalog_agent_tool]

orchestrator_agent    = create_tool_calling_agent(llm, orchestrator_tools, ORCHESTRATOR_PROMPT)
orchestrator_executor = AgentExecutor(
    agent    = orchestrator_agent,
    tools    = orchestrator_tools,
    verbose  = True,
    max_iterations = 12,
    handle_parsing_errors = True,
)

print("✅ Orchestratore assemblato.")
print("   Tool:", [t.name for t in orchestrator_tools])

✅ Orchestratore assemblato.
   Tool: ['get_employee_budget', 'agente_verifica_policy', 'agente_catalogo']


## 5. Test del Pattern Agent-as-Tool

### Scenario 1: Mario Rossi — ThinkPad P16s

In [7]:
print("=" * 60)
print("  AGENT-AS-TOOL — Scenario 1: Mario Rossi")
print("=" * 60)

r1 = orchestrator_executor.invoke({"input": (
    "Ciao! Sono Mario Rossi, badge E4521. "
    "Ho bisogno di un laptop per sviluppo software sul Project Delta. "
    "Posso acquistare il ThinkPad P16s?"
)})

print("\n📤 RISPOSTA FINALE:", r1["output"])

  AGENT-AS-TOOL — Scenario 1: Mario Rossi


> Entering new AgentExecutor chain...

Invoking: `get_employee_budget` with `{'employee_id': 'E4521'}`


{'status': 'OK', 'employee_id': 'E4521', 'name': 'Mario Rossi', 'department': 'Engineering', 'role': 'Senior Developer', 'budget_residuo_eur': 1800, 'active_project': 'Project Delta', 'manager_email': 'chiara.bianchi@azienda.it'}
Invoking: `agente_catalogo` with `ThinkPad P16s Gen 2`


Il ThinkPad P16s Gen 2 è disponibile nel catalogo con il codice LAP-DEV-01. Ecco le informazioni rilevanti:

* Nome: ThinkPad P16s Gen 2 / MacBook Pro M3
* Categoria: Laptop
* Ruoli target: Engineering, Data Science
* Prezzo: 1650€ (prezzo in eur)
* Specifiche: 32GB RAM, 1TB SSD, 16-inch display, GPU integrata/dedicata
* Disponibilità: Disponibile
Invoking: `agente_verifica_policy` with `Dipendente ruolo Senior Developer (Engineering), budget residuo 1800¢, vuole acquistare ThinkPad P16s Gen 2 a 1650¢.`


**CONTROLLO A — Limite fascia per ruolo:**
- Ruolo: S

### Scenario 2: Giulia Verdi — Monitor 4K

In [8]:
print("=" * 60)
print("  AGENT-AS-TOOL — Scenario 2: Giulia Verdi")
print("=" * 60)

r2 = orchestrator_executor.invoke({"input": (
    "Salve, sono Giulia Verdi (badge E1082), Marketing. "
    "Per la Brand Campaign Q2 ho bisogno del monitor 4K Dell UltraSharp. "
    "Posso acquistarlo?"
)})

print("\n📤 RISPOSTA FINALE:", r2["output"])

  AGENT-AS-TOOL — Scenario 2: Giulia Verdi


> Entering new AgentExecutor chain...

Invoking: `get_employee_budget` with `{'employee_id': 'E1082'}`


{'status': 'OK', 'employee_id': 'E1082', 'name': 'Giulia Verdi', 'department': 'Marketing', 'role': 'Content Specialist', 'budget_residuo_eur': 450, 'active_project': 'Brand Campaign Q2', 'manager_email': 'luca.neri@azienda.it'}
Invoking: `agente_catalogo` with `monitor 4K Dell UltraSharp`


Ecco i risultati per il monitor 4K Dell UltraSharp:

* Nome: Dell UltraSharp 27 4K (U2723QE)
* Prezzo: 520€
* Specifiche: 27-inch 4K IPS, USB-C Hub 90W power delivery
* Disponibilità: In stock
* Ruoli target: Tutti gli utenti

Non sono riuscito a trovare altri prodotti che corrispondano esattamente alla tua richiesta. Il monitor Dell UltraSharp 27 4K (U2723QE) sembra essere l'unico prodotto disponibile nel catalogo.
Invoking: `agente_verifica_policy` with `Dipendente ruolo Marketing, budget residuo 1500¢, vuole acquistare monitor 4K Dell UltraSharp a 

## 6. Confronto: Monolitico vs Agent-as-Tool

In [9]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║        CONFRONTO: AGENTE MONOLITICO vs AGENT-AS-TOOL (LangChain)        ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  MONOLITICO (Lab 2)                                                      ║
║  create_tool_calling_agent(llm, [budget, catalogo], prompt_con_policy)   ║
║  • 1 context window con tutto: policy + dati + catalogo                 ║
║  • PRO: semplice, 1 chiamata LLM per step                               ║
║  • CONTRO: context grande, rischio confusione cross-domain              ║
║                                                                          ║
║  AGENT-AS-TOOL (Lab 4)                                                   ║
║  Orchestratore usa Tool.from_function(AgentExecutor)                    ║
║  • Orchestratore: sa QUANDO delegare (non conosce i dettagli)           ║
║  • Policy Agent: solo testo policy, zero tool                           ║
║  • Catalog Agent: solo search_product_catalog, zero policy              ║
║  • PRO: separazione domini, meno allucinazioni trasversali              ║
║  • PRO: ogni agente modificabile indipendentemente                      ║
║  • CONTRO: N chiamate LLM, latenza maggiore                             ║
║                                                                          ║
║  QUANDO USARE AGENT-AS-TOOL?                                             ║
║  • Dominio complesso con conoscenze specialistiche separate             ║
║  • Policy che cambiano spesso (aggiorna 1 agente, non tutto)            ║
║  • Primo passo verso sistemi multi-agente (Giorno 4)                    ║
╚══════════════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════════════╗
║        CONFRONTO: AGENTE MONOLITICO vs AGENT-AS-TOOL (LangChain)        ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  MONOLITICO (Lab 2)                                                      ║
║  create_tool_calling_agent(llm, [budget, catalogo], prompt_con_policy)   ║
║  • 1 context window con tutto: policy + dati + catalogo                 ║
║  • PRO: semplice, 1 chiamata LLM per step                               ║
║  • CONTRO: context grande, rischio confusione cross-domain              ║
║                                                                          ║
║  AGENT-AS-TOOL (Lab 4)                                                   ║
║  Orchestratore usa Tool.from_function(AgentExecutor)                    ║
║  • Orchestratore: sa QUANDO delegare (non conosce i dettagli)           ║
║  •

## 7. Ispezione dei Tool Annidati

In [13]:
# Abilitiamo return_intermediate_steps per vedere le chiamate ai sotto-agenti
orchestrator_executor_detail = AgentExecutor(
    agent    = orchestrator_agent,
    tools    = orchestrator_tools,
    verbose  = False,
    max_iterations = 12,
    handle_parsing_errors = True,
    return_intermediate_steps = True,
)

result_detail = orchestrator_executor_detail.invoke({"input": (
    "Ciao, sono Mario Rossi (E4521). Voglio il ThinkPad P16s per Project Delta."
)})

print("=" * 60)
print("  TOOL CALL CHAIN — Agent-as-Tool")
print("=" * 60)
for i, (action, obs) in enumerate(result_detail["intermediate_steps"], 1):
    tool_type = "🤖 SubAgent" if action.tool in ["agente_verifica_policy", "agente_catalogo"] else "🔧 Tool"
    print(f"\n  Step {i}: {tool_type} '{action.tool}'")
    print(f"           Input: {str(action.tool_input)[:80]}")
    obs_preview = str(obs)[:120] + ("..." if len(str(obs)) > 120 else "")
    print(f"           Output: {obs_preview}")

  TOOL CALL CHAIN — Agent-as-Tool

  Step 1: 🔧 Tool 'get_employee_budget'
           Input: {'employee_id': 'E4521'}
           Output: {'status': 'OK', 'employee_id': 'E4521', 'name': 'Mario Rossi', 'department': 'Engineering', 'role': 'Senior Developer',...

  Step 2: 🤖 SubAgent 'agente_catalogo'
           Input: ThinkPad P16s
           Output: Il ThinkPad P16s Gen 2 / MacBook Pro M3 è un laptop disponibile in stock. Ecco le informazioni rilevanti:

* Nome: Think...

  Step 3: 🤖 SubAgent 'agente_verifica_policy'
           Input: Dipendente ruolo Senior Developer (Engineering), budget residuo 1800¢, vuole acq
           Output: **CONTROLLO A — Limite fascia per ruolo:**
- Ruolo: Senior Developer (Engineering)
- Limite laptop: 1800€
- Prezzo del p...


## 8. 💡 Esercizi Pratici

### Esercizio A — Terzo Sotto-Agente
Crea un terzo sotto-agente `agente_budget_analyzer` che:
- Usa `get_employee_budget`
- Classifica la capacità di acquisto del dipendente: ALTA (>1500€), MEDIA (500-1500€), BASSA (<500€)
- Wrappalo con `Tool.from_function()` e aggiungilo all'orchestratore

In [11]:
# ESERCIZIO A — Budget Analyzer Agent
BUDGET_AGENT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """TODO: sei un agente specializzato nell'analisi del budget dei dipendenti.
Usa 'get_employee_budget' per ottenere i dati e classifica la capacità di acquisto:
- ALTA: budget > 1500€
- MEDIA: budget 500-1500€
- BASSA: budget < 500€
Rispondi in italiano con budget esatto e classificazione."""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# TODO: Crea il budget_agent e budget_executor
# budget_agent    = create_tool_calling_agent(llm, [get_employee_budget], BUDGET_AGENT_PROMPT)
# budget_executor = AgentExecutor(agent=budget_agent, tools=[get_employee_budget], verbose=False)

# TODO: Wrappa come Tool
# budget_agent_tool = Tool.from_function(
#     func        = lambda q: budget_executor.invoke({"input": q})["output"],
#     name        = "agente_budget_analyzer",
#     description = "TODO: scrivi la descrizione"
# )

# TODO: Ricostruisci l'orchestratore con il terzo tool
# orchestrator_tools_v2 = [get_employee_budget, policy_agent_tool, catalog_agent_tool, budget_agent_tool]

print("💡 Decommenta e completa il codice per il terzo sotto-agente.")

💡 Decommenta e completa il codice per il terzo sotto-agente.


### Esercizio B — Agent-as-Tool + Human Approval
Integra il callback `HumanApprovalCallback` del Lab 3 nell'orchestratore.
Il flow completo: orchestratore → sotto-agenti → decisione → Human Approval se escalation.

In [12]:
# ESERCIZIO B — Combina Agent-as-Tool + Human Approval
# Suggerimento:
# 1. Aggiungi request_purchase_approval ai tool dell'orchestratore
# 2. Crea un HumanApprovalCallback (importa o ricopia dal Lab 3)
# 3. Passa callbacks=[approval_cb] all'AgentExecutor dell'orchestratore

print("💡 Combina i pattern dei Lab 3 e 4 per il flow completo.")

💡 Combina i pattern dei Lab 3 e 4 per il flow completo.


---
### ✅ Recap Lab 4

- Abbiamo costruito sotto-agenti specializzati (`AgentExecutor` con system prompt ristretto).
- Abbiamo usato **`Tool.from_function()`** per wrappare ogni `AgentExecutor` come tool LangChain standard.
- L'orchestratore usa i sotto-agenti **come qualsiasi altro tool** — nessuna API speciale.
- Abbiamo confrontato pattern monolitico vs Agent-as-Tool.
- Con `return_intermediate_steps=True` abbiamo visto l'intera catena di chiamate annidate.

**Questo è il ponte verso il Giorno 4**: il pattern Agent-as-Tool è il precursore dei sistemi multi-agente
con stato esplicito e routing condizionale (LangGraph).

➡️ **Prossimo notebook:** `05_Esercitazione_MVP.ipynb` — costruisci il tuo MVP con LangChain!